# Experiment 5
## Flow-Based Generative Model (Normalizing Flow)
**Aim:** Implement a simple flow-based generative model using PyTorch.

### Step 1 – Import Libraries

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

### Step 2 – Define an Affine Coupling Layer
A coupling layer is the building block of flow-based models. It transforms data invertibly.

In [ ]:
class AffineCouplingLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.scale_net  = nn.Sequential(nn.Linear(dim//2, 64), nn.ReLU(), nn.Linear(64, dim//2), nn.Tanh())
        self.shift_net  = nn.Sequential(nn.Linear(dim//2, 64), nn.ReLU(), nn.Linear(64, dim//2))

    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        s = self.scale_net(x1)
        t = self.shift_net(x1)
        y2 = x2 * torch.exp(s) + t
        log_det = s.sum(dim=1)
        return torch.cat([x1, y2], dim=1), log_det

    def inverse(self, y):
        y1, y2 = y.chunk(2, dim=1)
        s = self.scale_net(y1)
        t = self.shift_net(y1)
        x2 = (y2 - t) * torch.exp(-s)
        return torch.cat([y1, x2], dim=1)

### Step 3 – Stack Multiple Coupling Layers into a Flow Model

In [ ]:
class SimpleFlow(nn.Module):
    def __init__(self, dim, n_layers=4):
        super().__init__()
        self.layers = nn.ModuleList([AffineCouplingLayer(dim) for _ in range(n_layers)])

    def forward(self, x):
        log_det_total = 0
        for layer in self.layers:
            x, log_det = layer(x)
            log_det_total += log_det
        return x, log_det_total

    def sample(self, n):
        z = torch.randn(n, 2)      # sample from base Gaussian
        for layer in reversed(self.layers):
            z = layer.inverse(z)
        return z.detach()

### Step 4 – Generate 2D Target Data (Two Moons Shape)

In [ ]:
from sklearn.datasets import make_moons
X, _ = make_moons(n_samples=2000, noise=0.05)
data = torch.tensor(X, dtype=torch.float32)
print('Data shape:', data.shape)

### Step 5 – Train the Flow Model

In [ ]:
model   = SimpleFlow(dim=2, n_layers=4)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses  = []
for epoch in range(300):
    z, log_det = model(data)
    log_prob   = -0.5 * (z**2).sum(dim=1) - log_det   # negative log-likelihood
    loss       = log_prob.mean()
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if epoch % 50 == 0:
        losses.append(loss.item())
        print(f'Epoch {epoch:3d} | Loss: {loss.item():.4f}')

### Step 6 – Sample and Visualize

In [ ]:
samples = model.sample(1000).numpy()
plt.figure(figsize=(10, 4))
plt.subplot(1,2,1); plt.scatter(X[:,0], X[:,1], s=5, alpha=0.5); plt.title('Original Data')
plt.subplot(1,2,2); plt.scatter(samples[:,0], samples[:,1], s=5, alpha=0.5, color='orange'); plt.title('Flow-Generated Samples')
plt.tight_layout(); plt.show()

### Result
A normalizing flow model was implemented and trained. The model learned to map simple Gaussian samples to the target distribution.